# Dinomaly on CBIS-DDSM: full training run + report

Full training run of the Dinomaly anomaly detector on CBIS-DDSM, then a report:
training curves, the held-out metrics as they evolve, final numbers, score
distributions, ROC/PR curves, anomaly heatmaps, and a comparison against the
numbers from the Dinomaly paper.

**Setup.** Train on benign (normal) full mammograms only; score malignant as
anomalies at test time. DINOv2 ViT-B/14 (with registers) is frozen; only the MLP
bottleneck and the 8-block linear-attention decoder train. Config matches
Dinomaly's `dinomaly_mvtec_uni.py`: 448 resize / 392 crop, batch 16, StableAdamW
2e-3 -> 2e-4 warmup-cosine, 10k iters, hard-mining cosine loss.

Model / loss / training loop are in `src/`; this notebook drives the run.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, json, math
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

## 1. Run parameters

`DEV = True` shrinks the run so the notebook can be checked end to end in a few minutes. Set it to `False` for the real ~3 h run.

In [ ]:
DEV = False

BACKBONE = 'vit_base'
SEED = 1

if DEV:
    ITERS, EVAL_EVERY, WARMUP, HM_RAMP = 150, 150, 20, 150
else:
    ITERS, EVAL_EVERY, WARMUP, HM_RAMP = 10000, 2000, 100, 1000

from src.training.train_dinomaly import pick_device
device = pick_device()
print('device:', device, '| iters:', ITERS)

## 2. Data

In [ ]:
from src.data.cbis_ddsm import load_datasets

datasets = load_datasets()
sizes = {name: len(ds) for name, ds in datasets.items()}
sizes

In [ ]:
n_norm = sizes['train_normal'] + sizes['test_normal']
n_anom = sizes['test_anomaly']
print(f"normal (benign) images:   {n_norm:5d}  -> {sizes['train_normal']} train / {sizes['test_normal']} test")
print(f"anomaly (malignant):      {n_anom:5d}  -> all held out for test")
print(f"test set prevalence:      {n_anom / (sizes['test_normal'] + n_anom):.1%} anomaly")

## 3. Model

In [ ]:
from src.models.dinomaly import build_dinomaly, TARGET_LAYERS, FUSE_GROUPS, N_DECODER_BLOCKS

model, trainable = build_dinomaly(BACKBONE)
model = model.to(device)

frozen = sum(p.numel() for p in model.encoder.parameters())
train_p = sum(p.numel() for p in trainable.parameters())
print(f"backbone {BACKBONE}: {frozen/1e6:.1f}M frozen encoder params")
print(f"trainable (bottleneck + decoder): {train_p/1e6:.1f}M")
print(f"encoder layers tapped: {TARGET_LAYERS}")
print(f"fused into groups: {FUSE_GROUPS}")
print(f"decoder blocks: {N_DECODER_BLOCKS} (linear attention, eps-guarded)")
del model, trainable  # train() builds its own

## 4. Train

Live bar shows iter / epoch / loss / lr. Eval every `EVAL_EVERY` iters on the held-out normal + all malignant, checkpoint written each time. Interrupting the kernel keeps the last checkpoint.

In [ ]:
from src.training.train_dinomaly import TrainConfig, train

cfg = TrainConfig(
    backbone=BACKBONE, iters=ITERS, eval_every=EVAL_EVERY,
    warmup_iters=WARMUP, hm_ramp_iters=HM_RAMP, seed=SEED,
)
history = train(cfg, datasets=datasets, device=device)

## 5. Training curves

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.plot(history['iters'], history['loss'], color='tab:blue', lw=1.2)
ax1.set_xlabel('iteration'); ax1.set_ylabel('loss (cosine, hard-mined)', color='tab:blue')
ax1.set_title(f"Training loss  ({history['epochs']} epochs, {history['batches_per_epoch']} batches/epoch)")
ax2 = ax1.twinx()
ax2.plot(history['iters'], history['lr'], color='tab:orange', lw=1, alpha=0.5)
ax2.set_ylabel('lr', color='tab:orange')
for e in history['evals']:
    ax1.axvline(e['iter'], color='grey', ls=':', lw=0.8)
plt.tight_layout(); plt.show()

### Held-out metrics vs. training progress

In [ ]:
ev = pd.DataFrame(history['evals'])

fig, ax = plt.subplots(figsize=(11, 4))
for key, mark in [('auroc', 'o-'), ('ap', 's-'), ('f1_max', '^-')]:
    ax.plot(ev['iter'], ev[key], mark, label=key.upper())
ax.axhline(0.5, color='grey', ls='--', lw=0.8, label='chance AUROC')
ax.set_xlabel('iteration'); ax.set_ylabel('score'); ax.set_ylim(0, 1.02)
ax.legend(); ax.grid(alpha=0.3); ax.set_title('Image-level metrics during training')
plt.tight_layout(); plt.show()

ev[['iter', 'epoch', 'auroc', 'ap', 'f1_max']]

## 6. Final evaluation

Re-score the full held-out set from the trained model (per-image scores kept for the plots below).

In [ ]:
from torch.utils.data import ConcatDataset, DataLoader
from src.models.dinomaly import build_dinomaly
from src.eval.anomaly import score_dataset, metrics_from_scores

test_set = ConcatDataset([datasets['test_normal'], datasets['test_anomaly']])
test_loader = DataLoader(test_set, batch_size=16, shuffle=False, num_workers=4)

ckpt = torch.load(f"{history['out_dir']}/checkpoint.pt", map_location=device)
model, trainable = build_dinomaly(ckpt['config']['backbone'])
trainable.load_state_dict(ckpt['trainable'])
model = model.to(device).eval()

scores, labels = score_dataset(model, test_loader, device)
final = metrics_from_scores(scores, labels)
final

## 7. Score distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
bins = np.linspace(scores.min(), scores.max(), 50)
ax.hist(scores[labels == 0], bins=bins, alpha=0.6, label=f'normal (n={int((labels==0).sum())})', color='tab:green')
ax.hist(scores[labels == 1], bins=bins, alpha=0.6, label=f'malignant (n={int((labels==1).sum())})', color='tab:red')
ax.set_xlabel('image anomaly score  (mean of top 1% of the anomaly map)')
ax.set_ylabel('count'); ax.legend(); ax.set_title('Anomaly score by class')
plt.tight_layout(); plt.show()

print(f"normal    mean {scores[labels==0].mean():.4f}  median {np.median(scores[labels==0]):.4f}")
print(f"malignant mean {scores[labels==1].mean():.4f}  median {np.median(scores[labels==1]):.4f}")

## 8. ROC and precision-recall

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

fpr, tpr, _ = roc_curve(labels, scores)
prec, rec, _ = precision_recall_curve(labels, scores)
base_rate = labels.mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(fpr, tpr, lw=2); axes[0].plot([0, 1], [0, 1], 'k--', lw=0.8)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title(f"ROC  (AUROC {final['auroc']:.3f})")
axes[1].plot(rec, prec, lw=2); axes[1].axhline(base_rate, color='grey', ls='--', lw=0.8)
axes[1].set_xlabel('recall'); axes[1].set_ylabel('precision')
axes[1].set_title(f"PR  (AP {final['ap']:.3f}, base rate {base_rate:.2f})")
for a in axes: a.grid(alpha=0.3); a.set_xlim(0, 1); a.set_ylim(0, 1.02)
plt.tight_layout(); plt.show()

## 9. Anomaly heatmaps

Two confident normals, two confident malignants, and the worst false positive / false negative (normal scored high / malignant scored low).

In [ ]:
from src.data.cbis_ddsm import IMAGENET_MEAN, IMAGENET_STD
from src.eval.anomaly import anomaly_map_for_images

mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

order = np.argsort(scores)
normal_idx = [i for i in order if labels[i] == 0]
anom_idx = [i for i in order if labels[i] == 1]
picks = [
    ('normal, low score', normal_idx[0]),
    ('normal, low score', normal_idx[1]),
    ('malignant, high score', anom_idx[-1]),
    ('malignant, high score', anom_idx[-2]),
    ('normal, high score (FP)', normal_idx[-1]),
    ('malignant, low score (FN)', anom_idx[0]),
]

imgs = torch.stack([test_set[i][0] for _, i in picks])
amaps = anomaly_map_for_images(model, imgs, device)

fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for col, ((title, idx), img, amap) in enumerate(zip(picks, imgs, amaps)):
    disp = (img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    axes[0, col].imshow(disp); axes[0, col].set_title(f"{title}\nscore {scores[idx]:.3f}", fontsize=8)
    axes[1, col].imshow(disp)
    axes[1, col].imshow(amap, cmap='jet', alpha=0.45)
    for r in (0, 1): axes[r, col].axis('off')
axes[0, 0].set_ylabel('image', fontsize=9); axes[1, 0].set_ylabel('+ anomaly map', fontsize=9)
plt.tight_layout(); plt.show()

## 10. Comparison with the Dinomaly paper

The paper evaluates on industrial anomaly detection benchmarks in the multi-class
("unified") setting, with the same DINOv2-reg ViT-B/14 backbone and the same
training recipe used here. Numbers below are from the paper (Guo et al., CVPR
2025, arXiv:2405.14325).

In [ ]:
paper = pd.DataFrame([
    dict(dataset='MVTec-AD (paper)',  setting='multi-class, in-domain', backbone='ViT-B/14',
         I_AUROC=99.6, I_AP=99.8, I_F1max=99.0),
    dict(dataset='VisA (paper)',      setting='multi-class, in-domain', backbone='ViT-B/14',
         I_AUROC=98.7, I_AP=98.9, I_F1max=96.2),
    dict(dataset='Real-IAD (paper)',  setting='multi-class, in-domain', backbone='ViT-B/14',
         I_AUROC=89.3, I_AP=86.8, I_F1max=80.2),
    dict(dataset='CBIS-DDSM (ours)',  setting='benign-only train',      backbone='ViT-B/14',
         I_AUROC=round(final['auroc'] * 100, 1),
         I_AP=round(final['ap'] * 100, 1),
         I_F1max=round(final['f1_max'] * 100, 1)),
])
paper.set_index('dataset')

### Why this is context, not a target

The paper's benchmarks and CBIS-DDSM are different problems:

- **What "normal" and "anomaly" mean.** On MVTec/VisA/Real-IAD, normal is a clean
  manufactured part and an anomaly is a scratch, crack, or misprint: local,
  high-contrast, and absent from every training image. Here, normal is a *benign*
  mammogram that already contains masses and calcifications, and the anomaly is
  *malignancy* in similar-looking tissue. The model has to separate malignant
  from benign texture, not "defect present" from "defect absent".

- **Signal strength.** Industrial defects are visually obvious once localised.
  Malignancy on a mammogram is often subtle and diffuse, and benign vs. malignant
  is hard even for radiologists on the image alone.

- **In-domain vs. out-of-domain backbone.** DINOv2 was pretrained on natural
  images; industrial photos are closer to that distribution than grayscale x-ray.

- **No mammography baseline in the paper.** Dinomaly was never run on medical
  imaging, so there is no matched number to compare against. Our result is a
  first baseline for this framing on CBIS-DDSM, and the paper's Real-IAD row
  (its hardest benchmark, 89.3 AUROC) is the most useful reference point for how
  the method degrades as the task gets harder.

In [ ]:
print("SUMMARY")
print("=" * 60)
print(f"backbone            {BACKBONE}  (DINOv2-reg, frozen)")
print(f"train / test normal {sizes['train_normal']} / {sizes['test_normal']}")
print(f"test malignant      {sizes['test_anomaly']}")
print(f"iterations          {ITERS}  ({history['epochs']} epochs)")
print(f"wall time           {history['elapsed_min']:.0f} min on {device}")
print("-" * 60)
print(f"image-level AUROC    {final['auroc']:.4f}")
print(f"image-level AP       {final['ap']:.4f}   (base rate {labels.mean():.3f})")
print(f"image-level F1-max   {final['f1_max']:.4f}")
print("-" * 60)
print(f"checkpoint          {history['out_dir']}/checkpoint.pt")